In [12]:
import os
os.getcwd()


'D:\\MLP\\datasci'

In [13]:
os.chdir("D:/MLP/datasci")

In [14]:
os.getcwd()

'D:\\MLP\\datasci'

In [15]:

from pathlib import Path
from dataclasses import dataclass

from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories


In [16]:
@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [17]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self)-> DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir

        )
        return data_ingestion_config

In [ ]:
import os
import urllib.request as requests
from src.datascience import logger # logger not imported properly
import zipfile

In [19]:
## components of data ingestion


class DataIngestion:
    def __init__(self,
                 config: DataIngestionConfig,
                 ):
        self.config = config

    # Downloading ZIP file from source URL and saving it to local data file path
    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = requests.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"Data file already exists at {self.config.local_data_file}. Skipping download.")

    # Unzipping the downloaded file to the specified unzip directory
    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Data extracted successfully to {unzip_path}.")

In [20]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise

AttributeError: module 'src.datascience.logger' has no attribute 'info'

### check out the logging configuration